# Day 07 — Project: Wasel (Exercises)

Randa runs operations analytics at Wasel, the ride-hailing app expanding across Egypt. Leadership has a fixed budget for driver incentives next quarter and wants her recommendation on which cities actually need it — plus a read on whether the promo codes marketing keeps pushing are paying for themselves, and whether a coverage gap someone flagged in a meeting is real or a false alarm.

No new SQL syntax in this project: it's Days 01-05 (filtering, aggregation, JOINs, CASE, anti-joins, subqueries, CTEs, views, temp tables, window functions — everything except stored procedures, which SQLite has no equivalent for) applied to a completely different business from Rawaj. Nothing about the e-commerce schema carries over except the SQL.

## Getting connected

In [1]:
%load_ext sql

In [2]:
%sql sqlite:///../Databases/wasel.sqlite --alias wasel

Connecting to 'wasel'

A bare connection string, no SQLAlchemy needed — the same jupysql connection pattern Day 06 introduced.

In [3]:
%%sql
SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;

Running query in 'wasel'

name
cities
driver_city_coverage
drivers
promotions
ratings
riders
trips
vehicle_types
vehicles


9 tables — cities, drivers, riders, `driver_city_coverage` (the genuine N:N — a driver can cover more than one city), vehicle_types, vehicles, promotions, trips (the transactional core), and ratings. A quick preview of each before Randa's questions start:

In [4]:
from IPython.display import Markdown, display

tables_result = %sql SELECT name FROM sqlite_master WHERE type='table' ORDER BY name
table_names = tables_result.DataFrame()['name'].tolist()

for table in table_names:
    display(Markdown(f"### {table}"))
    preview = %sql SELECT * FROM {{table}} LIMIT 3
    display(preview)

Running query in 'wasel'

### cities

Running query in 'wasel'

city_id,city_name
1,Cairo
2,Giza
3,Alexandria


### driver_city_coverage

...

Running query in 'wasel'

type_id,type_name,base_fare,per_km_rate
1,Economy,12.0,3.5
2,Comfort,18.0,4.75
3,XL,25.0,6.0


### vehicles

Running query in 'wasel'

vehicle_id,driver_id,type_id,make,model,year,plate_number
1,1,1,Skoda,Rapid,2015,RNX 554
2,2,1,Chevrolet,Optra,2021,VSU 337
3,3,1,MG,5,2022,ZYE 800


---

Randa opens with the basics — how big is the business, and where does it actually happen.

### Q1 — How many trips has Wasel completed since launch, and how many were cancelled or a no-show?

**Business question:** the headline numbers for the leadership deck's first slide.

### Q2 — Which 5 cities have generated the most completed trips?

**Business question:** where the volume already is, before deciding where the *gaps* are.

### Q3 — What's the average completed-trip fare, company-wide?

**Business question:** a single reference number everything else in the deck gets compared against.

Next: whether two things marketing keeps asking about — promo codes and surge pricing — are actually significant, or just noise.

### Q4 — Which promo codes have actually been redeemed, and how many times each?

**Business question:** marketing wants to know which of this quarter's codes are pulling their weight before planning the next batch.

### Q5 — Are surge-priced trips a big share of the business, or a rare event?

**Business question:** riders complain about surge pricing loudly — worth checking whether it's actually common before it becomes a policy fight.

With the shape of the business established, Randa turns to the drivers themselves — who's carrying the business, and who isn't showing up at all.

### Q6 — Combine every completed trip with the driver who drove it, their vehicle, and the vehicle type.

**Business question:** the base join every driver-level question from here on builds on.

### Q7 — Which drivers have never completed a single trip?

**Business question:** the incentive budget is meant for drivers who are *active but under-earning* — not drivers who signed up and never drove. This anti-join tells Randa how many of those there are before she sizes the budget.

### Q8 — Bucket every driver into a 'top earner' (over 4,000 EGP total fares) or 'standard' tier.

**Business question:** a quick tier Randa can hand to finance without them re-deriving it.

### Q9 — Which cities see the highest share of cross-city trips?

**Business question:** cross-city rides (pickup and dropoff in different cities) take longer and tie up a driver further from home — worth knowing which cities lean on them most before setting incentive targets per city.

Two more precise questions need reusable definitions, not one-off filters — a subquery, then a saved view.

### Q10 — Which drivers earn above the company-wide average total fare?

**Business question:** "above average" is exactly the kind of threshold that should be computed, not hardcoded — the average shifts every quarter.

### Q11 — Save a 'top 20 drivers by earnings' view for the leadership deck.

**Business question:** this exact ranking gets pulled every quarterly review — define it once.

### Q12 — Which cities average a longer completed-trip distance than the network as a whole?

**Business question:** longer average trips mean a driver is off the road (and off the meter, between fares) longer per ride — a CTE keeps the "network average" and the per-city comparison in one readable query.

### Q13 — Materialize the most recently completed quarter's trips into a temp table, then pull two different breakdowns from it.

**Business question:** Randa needs both a by-city and a by-payment-method cut of the same quarter's data — materialize the filtered rows once instead of repeating the date filter twice. (Data runs through May 2025, so Q1 2025 — Jan through Mar — is the most recent full quarter; Q2 is still in progress.)

Randa's last section is about drivers again — but ranked and tracked over time, not just totaled.

### Q14 — Rank drivers by completed-trip count within their own home city.

**Business question:** "top driver" only means something relative to local competition — a driver in a small city with 30 trips might be the best there, even if that's fewer than the average driver in Cairo.

### Q15 — Running total of completed trips, company-wide, month over month.

**Business question:** a growth curve for the leadership deck's second slide.

### Q16 — Is month-over-month trip growth accelerating or slowing?

**Business question:** the running total always goes up — what leadership actually wants to know is whether the *rate* is holding.

### Q17 — Each top-earning driver's single best day.

**Business question:** if Wasel wants to feature a "driver of the month" story, this is where to find one worth telling.

Last question set, before Randa signs off: is the rating data itself trustworthy enough to lean on?

### Q18 — What's the average rating riders give drivers, versus drivers give riders?

**Business question:** if one side rates noticeably harsher than the other, that's worth knowing before using ratings to justify a driver's incentive tier.

### Q19 — How many completed trips have no rating at all, from either side?

**Business question:** ratings only mean something if there's enough of them — worth flagging the actual coverage before anyone builds a policy on top of "average rating".

### Q20 — How many cities have driver coverage but zero completed trips there?

**Business question:** the coverage gap someone flagged in a meeting, checked properly — a city can have a driver "based" there on paper and still have no real activity.

**Q20b — Confirm it: is this a demand problem or a data problem?**

## Your turn

That's all 20 questions. Once you've worked through them, check your answers and read Randa's actual recommendation in `project_07_solutions.ipynb` — including which of two plausible-looking city candidates the data does *not* actually support.